# Multimodal Laya — RLCD training

Trains the **bridge only**: a projector and two cross-attention blocks that let
Laya's existing decision head read SigLIP patch features. Both towers stay
frozen, so this is ~30M trainable parameters.

Before running, you need:

1. `data/vqa/` — VQA v2 questions + annotations ([visualqa.org](https://visualqa.org/download.html))
2. `cache/siglip_*.pt` — `python3 scripts/extract_features.py` output
3. A baseline number from `scripts/baseline_text_only.py` — **get this first.**
   Without a floor you cannot tell whether the bridge did anything.

## 0. Colab setup

Skip this cell if you are running locally.

Sized for **free-tier T4**: 16 GB VRAM, ~12 GB RAM, ~107 GB disk, and a session
that will disconnect on you. The two things that actually bite are RAM (features
are memory-mapped, never loaded whole) and disk (cache a subset of images, not
all of COCO — see the table in `scripts/extract_features.py`).

Mount Drive if you want the feature cache to survive a disconnect; re-extracting
20k images costs ~10 minutes, re-downloading COCO costs a lot more.

In [ ]:
COLAB = True
if COLAB:
    !pip -q install torch transformers safetensors huggingface_hub laya pillow
    from google.colab import drive; drive.mount('/content/drive')

    !git clone -q https://github.com/NandhaKishorM/laya /content/laya_repo 2>/dev/null || true
    %cd /content
    # put THIS project's src/ + scripts/ on the path (upload or clone your copy)
    import sys; sys.path.insert(0, "/content/laya-multimodal")

    # --- data ---
    !mkdir -p data/vqa data/coco cache
    !wget -q -c https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip -P data/vqa
    !wget -q -c https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip -P data/vqa
    !cd data/vqa && unzip -oq '*.zip'
    !wget -q -c http://images.cocodataset.org/zips/train2014.zip -P data/coco   # ~13 GB
    !cd data/coco && unzip -oq train2014.zip

    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    !df -h /content | tail -1
    !free -g | head -2

## 0b. Environment

In [ ]:
!pip -q install torch transformers safetensors huggingface_hub laya

import os, sys, json, math, time, random
import torch, torch.nn as nn
# Resolve the repo root whichever way this is launched: interactively from
# notebooks/, or headless via nbconvert from the repo root.
ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
sys.path.insert(0, ROOT)

DEV = ("cuda" if torch.cuda.is_available() else
       "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEV, "|", torch.cuda.get_device_name(0) if DEV=="cuda" else "")
torch.manual_seed(0); random.seed(0)

VQA_DIR   = os.path.join(ROOT, "data/vqa")
FEAT_PATH = os.path.join(ROOT, "cache/siglip_train2014")   # .npy/.json stem
OUT_DIR   = os.path.join(ROOT, "checkpoints/mm_laya_v1")

### Extract features for a subset

Full COCO at base-224 is ~25 GB of features; at so400m-384 it is ~139 GB. On a
free runtime, cap it. 20k images at base-224 is ~6 GB and gives roughly 40k
yes/no questions — plenty to see whether the bridge learns anything.

Run this **once**, then copy the two files to Drive so a disconnect does not
cost you the extraction.

In [ ]:
!python3 /content/laya-multimodal/scripts/extract_features.py \
    --images data/coco/train2014 \
    --out cache/siglip_train2014 \
    --model google/siglip-base-patch16-224 \
    --max-images 20000 --batch 32

# survive a disconnect
# !cp cache/siglip_train2014.* /content/drive/MyDrive/

## 1. Model — Laya's head, a new bridge

In [ ]:
from src.model import build

# d_v must match your cached features: so400m=1152, base=768.
# extract_features.py prints it at the end.
mm, tok, cfg = build(repo="convaiinnovations/laya", d_v=768,   # base-224=768, so400m=1152 (extract prints it)
                     cross_layers=2, freeze_text=True)
mm = mm.to(DEV)
mm.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

## 2. Data

`build_items` turns each VQA annotation into a Laya item. The target is the
**distribution over 10 human answers**, not a hard label — 8 yes / 2 no becomes
`[0.2, 0.8]`. That is what makes the proper-scoring-rule reward meaningful here.

In [ ]:
from src.data import build_items, split_items, collate, FeatureStore

items = build_items(VQA_DIR, tok, cfg, split="train2014",
                    kinds=("noul",),        # start with one primitive
                    limit=None)
train_items, val_items = split_items(items, val_frac=0.05)

# memory-mapped: opening this costs ~nothing, rows page in per batch.
# A dict from torch.load would put the whole cache in RAM and OOM Colab.
feats = FeatureStore(FEAT_PATH)
train_items = [i for i in train_items if i["image_id"] in feats]
val_items   = [i for i in val_items   if i["image_id"] in feats]
print("%d train / %d val items have cached features" % (len(train_items), len(val_items)))
assert feats.d_v == mm.d_v, "d_v mismatch — rebuild with d_v=%d" % feats.d_v

## 3. RLCD

Same recipe as text Laya:

1. sample `G` noisy logit distributions, **zero-mean projected** (softmax is
   shift-invariant, so un-projected noise wastes exploration)
2. score them with `proper_reward` — log + spherical, plus RPS on `score` questions
3. the group's own mean is the baseline — **no value network**
4. policy gradient, plus a full-weight soft cross-entropy anchor

The CE term does the learning; the RL term shapes calibration. Pure policy
gradient at this data scale is far too noisy on its own.

In [ ]:
from laya.common import proper_reward

EPOCHS, MICRO, ACCUM, GROUP = 3, 16, 2, 4
LR_BRIDGE, LR_HEAD = 2.0e-4, 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1
W_SPH, W_RPS, W_CE = 0.75, 1.0, 1.0

bridge = list(mm.projector.parameters()) + list(mm.cross.parameters())
headp  = [p for p in list(mm.head.parameters()) + list(mm.scorer.parameters())
          if p.requires_grad]
opt = torch.optim.AdamW([{"params": bridge, "lr": LR_BRIDGE},
                         {"params": headp,  "lr": LR_HEAD}], weight_decay=0.01)
steps = (len(train_items) // (MICRO*ACCUM)) * EPOCHS
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1,steps), eta_min=1e-6)
# A10G/A100 are Ampere+ and support bf16: wider range, no loss-scale tuning.
AMP_DTYPE = torch.bfloat16 if (DEV=="cuda" and torch.cuda.get_device_capability(0)[0] >= 8) \
            else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(DEV=="cuda" and AMP_DTYPE is torch.float16))
print("autocast dtype:", AMP_DTYPE)
print("%d optimizer steps over %d epochs" % (steps, EPOCHS))

def batch_to(chunk):
    b = collate(chunk, tok.pad_token_id, feats)
    return {k: (v.to(DEV) if torch.is_tensor(v) else v) for k, v in b.items()}

## 3b. Eval helpers

Defined before training so the loop can call them for periodic validation.

In [ ]:
from laya.common import confidence_from_probs

def evaluate(split, temps=None):
    mm.eval(); C, K_, Z = [], [], []
    with torch.no_grad():
        for s in range(0, len(split), 64):
            chunk = split[s:s+64]; b = batch_to(chunk)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEV=="cuda")):
                lg = mm(b["input_ids"], b["attention_mask"], b["marker_pos"],
                        b["marker_mask"], b["qtype"], b["image_feats"].float())
            lg = lg.float().cpu()
            for r, it in enumerate(chunk):
                k = len(it["markers"]); Z.append(lg[r,:k])
                K_.append(it); C.append(int(max(range(k), key=lambda j: it["target"][j])))
    accs, confs = [], []
    for z, it, g in zip(Z, K_, C):
        t = 1.0 if temps is None else temps[it["qtype"]]
        p = torch.softmax(z/max(1e-3,t), -1).numpy()
        accs.append(1.0 if int(p.argmax())==g else 0.0)
        confs.append(confidence_from_probs(p, len(p)))
    return accs, confs, Z, K_, C

def ece(conf, corr, bins=15):
    e, n = 0.0, len(conf)
    for b in range(bins):
        lo, hi = b/bins, (b+1)/bins
        sel = [i for i in range(n) if lo < conf[i] <= hi]
        if sel: e += (len(sel)/n)*abs(sum(conf[i] for i in sel)/len(sel)
                                      - sum(corr[i] for i in sel)/len(sel))
    return e

accs, confs, Z, K_, C = evaluate(val_items)
print("val accuracy %.4f | ECE %.4f | mean conf %.4f"
      % (sum(accs)/len(accs), ece(confs,accs), sum(confs)/len(confs)))
print("\n  %-10s %9s %10s" % ("threshold","coverage","accuracy"))
for t in (0.0,0.3,0.5,0.7,0.9):
    sel=[i for i in range(len(confs)) if confs[i]>=t]
    if sel: print("  %-10.2f %8.1f%% %10.4f" % (t, 100*len(sel)/len(confs),
                                                sum(accs[i] for i in sel)/len(sel)))
print("\ncompare against scripts/baseline_text_only.py — that is the floor.")

In [ ]:
import json

CKPT_DIR   = OUT_DIR; os.makedirs(CKPT_DIR, exist_ok=True)
LATEST     = os.path.join(CKPT_DIR, "latest.pt")
HISTORY    = os.path.join(CKPT_DIR, "history.jsonl")
LOG_EVERY, SAVE_EVERY, EVAL_EVERY = 50, 500, 2000
SHUFFLE_SEED = 1234

def trainable_state():
    return {"projector": mm.projector.state_dict(), "cross": mm.cross.state_dict(),
            "head": mm.head.state_dict(), "scorer": mm.scorer.state_dict()}

def save_ckpt(epoch, step_in_epoch, gstep, tag="latest"):
    """Write to a temp file then rename. A spot interruption mid-write would
    otherwise leave a truncated checkpoint, which is worse than none."""
    tmp = os.path.join(CKPT_DIR, "_tmp.pt")
    torch.save({**trainable_state(),
                "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": gstep,
                "d_v": mm.d_v, "cross_layers": len(mm.cross),
                "config": {"EPOCHS":EPOCHS,"MICRO":MICRO,"ACCUM":ACCUM,"GROUP":GROUP,
                           "LR_BRIDGE":LR_BRIDGE,"LR_HEAD":LR_HEAD,"W_CE":W_CE}}, tmp)
    os.replace(tmp, os.path.join(CKPT_DIR, tag + ".pt"))

def log(rec):
    with open(HISTORY, "a") as f:
        f.write(json.dumps(rec) + "\n")

# ---- resume, if a previous run was interrupted -------------------------------
start_epoch, start_step, gstep = 0, 0, 0
if os.path.exists(LATEST):
    ck = torch.load(LATEST, map_location=DEV)
    mm.projector.load_state_dict(ck["projector"]); mm.cross.load_state_dict(ck["cross"])
    mm.head.load_state_dict(ck["head"]);           mm.scorer.load_state_dict(ck["scorer"])
    opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch, start_step, gstep = ck["epoch"], ck["step_in_epoch"], ck["global_step"]
    print("RESUMED from epoch %d, item %d (global step %d)" % (start_epoch, start_step, gstep))
else:
    open(HISTORY, "w").close()
    print("fresh run")

best_val = -1.0
t0 = time.time()

for epoch in range(start_epoch, EPOCHS):
    mm.train()
    # Deterministic per-epoch order, so resuming mid-epoch lands on the same
    # items a fresh run would have seen.
    order = list(range(len(train_items)))
    random.Random(SHUFFLE_SEED + epoch).shuffle(order)
    begin = start_step if epoch == start_epoch else 0

    sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * (epoch / max(1, EPOCHS - 1))
    opt.zero_grad(set_to_none=True)
    run, nb_, accum = 0.0, 0, 0

    for s in range(begin, len(order), MICRO):
        chunk = [train_items[i] for i in order[s:s + MICRO]]
        if not chunk:
            continue
        b = batch_to(chunk)
        mask, target = b["marker_mask"], b["target"]

        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
            logits = mm(b["input_ids"], b["attention_mask"], b["marker_pos"],
                        mask, b["qtype"], b["image_feats"].float())
        logits = logits.float()
        k = mask.sum(-1, keepdim=True).float()

        eps = torch.randn((GROUP,) + logits.shape, device=DEV) * sigma * mask
        eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
        z = logits.detach().unsqueeze(0) + eps
        q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

        with torch.no_grad():
            r = proper_reward(q, target.unsqueeze(0), b["qtype"], mask,
                              w_sph=W_SPH, w_rps=W_RPS)
            adv = r - r.mean(0, keepdim=True)
            adv = adv / (adv.std() + 1e-6)

        logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
        loss_rl = -(adv * logp).mean()
        loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
        loss = (loss_rl + W_CE * loss_ce) / ACCUM

        scaler.scale(loss).backward(); accum += 1
        if accum % ACCUM == 0 or (s + MICRO) >= len(order):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in mm.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)

        run += loss.item() * ACCUM; nb_ += 1; gstep += 1

        if gstep % LOG_EVERY == 0:
            rec = {"t": round(time.time()-t0,1), "epoch": epoch, "step": gstep,
                   "loss": round(run/max(1,nb_),4), "loss_rl": round(loss_rl.item(),4),
                   "loss_ce": round(loss_ce.item(),4), "reward": round(r.mean().item(),4),
                   "lr": sched.get_last_lr()[0], "sigma": round(sigma,3)}
            log(rec)
            print("  ep%d step%-7d loss %.4f  ce %.4f  reward %.3f  lr %.2e  %.0fs"
                  % (epoch, gstep, rec["loss"], rec["loss_ce"], rec["reward"],
                     rec["lr"], time.time()-t0))

        if gstep % SAVE_EVERY == 0:
            save_ckpt(epoch, s + MICRO, gstep)

        if gstep % EVAL_EVERY == 0:
            a_, c_, *_ = evaluate(val_items[:2000])
            acc, e_ = sum(a_)/len(a_), ece(c_, a_)
            log({"t": round(time.time()-t0,1), "epoch": epoch, "step": gstep,
                 "val_acc": round(acc,4), "val_ece": round(e_,4)})
            print("    val@%d  acc %.4f  ECE %.4f" % (gstep, acc, e_))
            if acc > best_val:
                best_val = acc; save_ckpt(epoch, s + MICRO, gstep, tag="best")
                print("    new best -> best.pt")
            mm.train()

    save_ckpt(epoch + 1, 0, gstep)
    print("=== epoch %d done | avg loss %.4f | %.0fs ===" % (epoch, run/max(1,nb_), time.time()-t0))
    start_step = 0

save_ckpt(EPOCHS, 0, gstep, tag="final")
print("\\ndone. checkpoints: latest.pt / best.pt / final.pt   history: history.jsonl")

## 3c. Curves

Read from `history.jsonl`, so this works mid-run in another kernel, or after an
interruption, without re-training anything.

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(HISTORY) if l.strip()]
tr = [r for r in rows if "loss" in r]
va = [r for r in rows if "val_acc" in r]
print("%d train points, %d val points" % (len(tr), len(va)))

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("multimodal Laya — bridge training", fontsize=12)

ax[0,0].plot([r["step"] for r in tr], [r["loss"] for r in tr], lw=1)
ax[0,0].set_title("total loss"); ax[0,0].set_xlabel("step")

ax[0,1].plot([r["step"] for r in tr], [r["loss_ce"] for r in tr], lw=1, label="CE")
ax[0,1].plot([r["step"] for r in tr], [r["loss_rl"] for r in tr], lw=1, label="RL")
ax[0,1].legend(); ax[0,1].set_title("loss split — CE learns, RL calibrates")
ax[0,1].set_xlabel("step")

ax[1,0].plot([r["step"] for r in tr], [r["reward"] for r in tr], lw=1, color="tab:green")
ax[1,0].set_title("proper-scoring reward (higher is better)"); ax[1,0].set_xlabel("step")

if va:
    a2 = ax[1,1]
    a2.plot([r["step"] for r in va], [r["val_acc"] for r in va], "o-", label="val accuracy")
    a2.set_xlabel("step"); a2.set_ylabel("accuracy")
    a3 = a2.twinx()
    a3.plot([r["step"] for r in va], [r["val_ece"] for r in va], "s--",
            color="tab:red", label="val ECE")
    a3.set_ylabel("ECE (lower better)")
    a2.set_title("validation — watch these diverge")
    a2.legend(loc="lower left"); a3.legend(loc="upper right")
else:
    ax[1,1].text(.5,.5,"no val points yet\n(EVAL_EVERY=%d)" % EVAL_EVERY,
                 ha="center", va="center"); ax[1,1].axis("off")

for a in ax.flat:
    a.grid(alpha=.3)
plt.tight_layout()
out = os.path.join(CKPT_DIR, "curves.png")
plt.savefig(out, dpi=130, bbox_inches="tight")
print("saved", out)
plt.show()

if va:
    print("\nAccuracy rising while ECE also rises means the model is getting more")
    print("right AND more overconfident. That is the temperature fit's job, not a")
    print("reason to stop training.")

## 4. Final evaluation

Full validation set, uncalibrated. The number to compare against is
`scripts/baseline_text_only.py` — without that floor a multimodal accuracy
means nothing on its own.

In [ ]:
# load the best checkpoint rather than whatever the last step happened to leave
best = os.path.join(CKPT_DIR, "best.pt")
if os.path.exists(best):
    ck = torch.load(best, map_location=DEV)
    mm.projector.load_state_dict(ck["projector"]); mm.cross.load_state_dict(ck["cross"])
    mm.head.load_state_dict(ck["head"]);           mm.scorer.load_state_dict(ck["scorer"])
    print("evaluating best.pt (step %d)" % ck["global_step"])

accs, confs, Z, K_, C = evaluate(val_items)
print("\nval accuracy %.4f | ECE %.4f | mean conf %.4f"
      % (sum(accs)/len(accs), ece(confs, accs), sum(confs)/len(confs)))

print("\n  %-10s %9s %10s" % ("threshold", "coverage", "accuracy"))
for t in (0.0, 0.3, 0.5, 0.7, 0.9):
    sel = [i for i in range(len(confs)) if confs[i] >= t]
    if sel:
        print("  %-10.2f %8.1f%% %10.4f"
              % (t, 100*len(sel)/len(confs), sum(accs[i] for i in sel)/len(sel)))

print("\ncoverage at fixed precision is the headline, not raw accuracy —")
print("it is what a calibrated probability buys you.")

## 5. Fit temperatures, then save

In [ ]:
def fit_temp(pairs):
    if len(pairs) < 25: return 1.0
    best, bt = float("inf"), 1.0
    for t in [0.2*(1.06**i) for i in range(70)]:
        tot = 0.0
        for z, tgt in pairs:
            lp = torch.log_softmax(z/t, -1)
            tot += -(torch.tensor(tgt)*lp).sum().item()
        if tot < best: best, bt = tot, t
    return round(bt, 4)

temps = [1.0, 1.0, 1.0]
for qt in range(3):
    sel = [(z, it["target"]) for z, it in zip(Z, K_) if it["qtype"] == qt]
    if sel: temps[qt] = fit_temp(sel)
print("fitted temperatures (choice, score, noul):", temps)

accs2, confs2, *_ = evaluate(val_items, temps)
print("after temperature: accuracy %.4f (unchanged by design) | ECE %.4f -> %.4f"
      % (sum(accs2)/len(accs2), ece(confs,accs), ece(confs2,accs2)))

os.makedirs(OUT_DIR, exist_ok=True)
torch.save({"projector": mm.projector.state_dict(),
            "cross": mm.cross.state_dict(),
            "head": mm.head.state_dict(),
            "scorer": mm.scorer.state_dict(),
            "temperature": temps, "d_v": mm.d_v,
            "cross_layers": len(mm.cross)},
           os.path.join(OUT_DIR, "bridge.pt"))
print("saved bridge to", OUT_DIR, "— towers are frozen, so only this needs storing")